In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from etl.banco import conectar
from sqlalchemy import text
conn = conectar()
print('conectado')

In [ ]:
# Pergunta 1 - 5 orgaos maior custo total
sql = "SELECT nome_orgao_superior, SUM(valor_total) as custo_total FROM silver_viagem GROUP BY nome_orgao_superior ORDER BY custo_total DESC LIMIT 5"
df1 = pd.read_sql(sql, conn)
df1

In [ ]:
plt.figure(figsize=(10,5))
sns.barplot(x='custo_total', y='nome_orgao_superior', data=df1)
plt.title('Top 5 Orgaos - Maior Custo Total')
plt.xlabel('Custo Total')
plt.ylabel('Orgao')
plt.show()

In [ ]:
# Pergunta 2 e 3
df2 = pd.read_sql("SELECT destinos, AVG(valor_total) as custo_medio FROM silver_viagem GROUP BY destinos ORDER BY custo_medio DESC LIMIT 3", conn)
df3 = pd.read_sql("SELECT id_viagem, nome_orgao_superior, duracao_dias, valor_total FROM silver_viagem ORDER BY duracao_dias DESC LIMIT 1", conn)
display(df2)
display(df3)

In [ ]:
# Gold - tabela agregada JOIN + GROUP BY
conn.execute(text("DROP TABLE IF EXISTS gold_custo_orgao"))
conn.execute(text("CREATE TABLE gold_custo_orgao AS SELECT v.nome_orgao_superior, COUNT(p.id_pagamento) as qtd_pagamentos, SUM(p.valor) as total_pago FROM silver_viagem v JOIN silver_pagamento p ON v.id_viagem = p.id_viagem GROUP BY v.nome_orgao_superior"))
conn.commit()
print('gold_custo_orgao criada')

In [ ]:
# Gold - VIEW
conn.execute(text("CREATE OR REPLACE VIEW vw_gold_custo_orgao AS SELECT * FROM gold_custo_orgao ORDER BY total_pago DESC"))
conn.commit()
print('VIEW criada')

In [ ]:
# Restante perguntas + graficos Gold
q4 = pd.read_sql("SELECT tipo_pagamento, AVG(valor) as media FROM silver_pagamento GROUP BY tipo_pagamento ORDER BY media DESC LIMIT 1", conn)
q5 = pd.read_sql("SELECT meio_transporte, COUNT(*) as total FROM silver_trecho GROUP BY meio_transporte ORDER BY total DESC LIMIT 5", conn)
q6 = pd.read_sql("SELECT destino_uf, COUNT(*) as total FROM silver_trecho GROUP BY destino_uf ORDER BY total DESC LIMIT 5", conn)
q7 = pd.read_sql("SELECT nome_orgao_pagador, SUM(valor) as total FROM silver_pagamento GROUP BY nome_orgao_pagador ORDER BY total DESC LIMIT 1", conn)
fig, ax = plt.subplots(1,3, figsize=(18,5))
sns.barplot(x='total', y='meio_transporte', data=q5, ax=ax[0])
ax[0].set_title('Meio Transporte Mais Usado')
sns.barplot(x='total', y='destino_uf', data=q6, ax=ax[1])
ax[1].set_title('UF Destino Mais Frequente')
sns.barplot(x='total_pago', y='nome_orgao_superior', data=pd.read_sql('SELECT * FROM gold_custo_orgao ORDER BY total_pago DESC LIMIT 5', conn), ax=ax[2])
ax[2].set_title('Gold - Orgao que Mais Pagou')
plt.tight_layout()
plt.show()